In [7]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate, PhaseEstimation
from qiskit import transpile
from qiskit_aer import AerSimulator
import numpy as np


In [8]:
def haar_random_unitary(d: int, rng=None) -> np.ndarray:
    rng = rng or np.random.default_rng()
    z = (rng.normal(size=(d, d)) + 1j * rng.normal(size=(d, d))) / np.sqrt(2.0)
    q, r = np.linalg.qr(z)
    diag = np.diag(r)
    phases = diag / np.abs(diag)
    phases[np.abs(diag) == 0] = 1.0
    return q * phases

def haar_unitary_circuit(n_qubits: int, rng=None, label='U') -> QuantumCircuit:
    """
    Generate a Haar random unitary as a Qiskit QuantumCircuit.

    Args:
        n_qubits: Number of qubits (matrix dimension = 2^n_qubits)
        rng: Optional numpy random Generator
        label: Label for the gate

    Returns:
        A QuantumCircuit implementing the Haar random unitary
    """
    d = 2 ** n_qubits
    U_matrix = haar_random_unitary(d, rng)

    # Wrap matrix as a Qiskit gate
    gate = UnitaryGate(U_matrix, label=label)

    # Build a circuit from it
    qc = QuantumCircuit(n_qubits, name=label)
    qc.append(gate, range(n_qubits))

    return qc

In [11]:
n_ancilla = 6
n_target = 3

ancilla1 = QuantumRegister(n_ancilla, 'anc1')
ancilla2 = QuantumRegister(n_ancilla, 'anc2')
target = QuantumRegister(n_target, 'tgt')
c1 = ClassicalRegister(n_ancilla, 'c1')
c2 = ClassicalRegister(n_ancilla, 'c2')

qc = QuantumCircuit(ancilla1, ancilla2, target, c1, c2)

In [12]:
rng1 = np.random.default_rng(seed=42)
rng2 = np.random.default_rng(seed =43) # fix seed for reproducibility
U1_circuit = haar_unitary_circuit(n_target, rng=rng1, label='U1')
U2_circuit = haar_unitary_circuit(n_target, rng=rng2, label='U2')

# Build QPE circuits using the same U
qpe1 = PhaseEstimation(num_evaluation_qubits=n_ancilla, unitary=U1_circuit)
qpe2 = PhaseEstimation(num_evaluation_qubits=n_ancilla, unitary=U2_circuit)

# Full circuit
anc1   = QuantumRegister(n_ancilla, 'anc1')
anc2   = QuantumRegister(n_ancilla, 'anc2')
target = QuantumRegister(n_target,  'target')
c1     = ClassicalRegister(n_ancilla, 'c1')
c2     = ClassicalRegister(n_ancilla, 'c2')

qc = QuantumCircuit(anc1, anc2, target, c1, c2)

# Prepare a random target state
qc.h(target)

# QPE 1 → measure → QPE 2 → measure
qc.compose(qpe1, qubits=list(anc1) + list(target), inplace=True)
qc.measure(anc1, c1)
qc.compose(qpe2, qubits=list(anc2) + list(target), inplace=True)
qc.measure(anc2, c2)

# Run
sim = AerSimulator()
transpiled_qc = transpile(qc, sim)
result = sim.run(transpiled_qc, shots=1000).result()
counts = result.get_counts()

# Verify
delta = 1
verified = sum(
    count for bits, count in counts.items()
    for c2_bits, c1_bits in [bits.split(' ')]
    if abs(int(c1_bits, 2) - int(c2_bits, 2)) <= delta
)
print(f"Verification rate: {verified / sum(counts.values()):.3f}")

/var/folders/ll/9dhmfhz932g_qjcmrdtyvjjm0000gn/T/ipykernel_86704/3725321019.py:7: DeprecationWarning: The class ``qiskit.circuit.library.phase_estimation.PhaseEstimation`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use qiskit.circuit.library.phase_estimation instead.
  qpe1 = PhaseEstimation(num_evaluation_qubits=n_ancilla, unitary=U1_circuit)
/var/folders/ll/9dhmfhz932g_qjcmrdtyvjjm0000gn/T/ipykernel_86704/3725321019.py:8: DeprecationWarning: The class ``qiskit.circuit.library.phase_estimation.PhaseEstimation`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use qiskit.circuit.library.phase_estimation instead.
  qpe2 = PhaseEstimation(num_evaluation_qubits=n_ancilla, unitary=U2_circuit)


Verification rate: 0.041
